In [14]:
import sys
sys.path.append("./lsr_package")
from lsr.transformer import LSR
import pandas as pd
from pyterrier_pisa import PisaIndex
import pickle
import pyterrier as pt
if not pt.started():
    pt.init()
from ir_measures import *
import os
import numpy as np

/tmp/ipykernel_7311/1103427932.py:8: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


In [15]:
docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")
training_docs = pd.read_pickle("/nfs/primary/sas_reranker/d2qmm_ohsumed_training_data.pkl")

def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

In [22]:
import pyterrier_dr
model_folder = "/nfs/primary/SPLADE/splade_class/lsr_package/outputs/splade_ohsumed_multiple_negative/model/"
index_folder = "/nfs/primary/SPLADE/splade_class/indices/splade_ohsumed_multiple_negative.flex"
lsr = LSR(model_folder).doc_encoder(sparse = False)
index = pyterrier_dr.FlexIndex(index_folder)
pipe = lsr >> index
gen_docs = (d for d in docs.to_dict(orient="records"))

pipe.index(gen_docs)

indexing: 14430dvec [01:30, 159.81dvec/s]


FlexIndex('/nfs/primary/SPLADE/splade_class/indices/splade_ohsumed_multiple_negative.flex')

In [4]:
for file in os.listdir("."):
    if ".pkl" in file and "cache" not in file and "vocab" not in file:
        print(file)

svm_coefficients_v3.pkl
svm_coefficients.pkl
trained_parameter_all_data.pkl
trained_parameter_no_regularisation.pkl
trained_parameter_0.1_l1reg.pkl
trained_parameter_0.2_l1reg.pkl
trained_parameter_0.3_l1reg.pkl
trained_parameter_0.4_l1reg.pkl
trained_parameter_0.5_l1reg.pkl
trained_parameter_0.6_l1reg.pkl
trained_parameter_0.7_l1reg.pkl
trained_parameter_0.8_l1reg.pkl
trained_parameter_0.9_l1reg.pkl
trained_parameter_10_l1reg.pkl
trained_no_sigmoid_on_scores.pkl
trained_no_sigmoid_on_scores_no_sigmoid_param.pkl
svm_coefficients_v1.pkl
svm_coefficients_v4.pkl
trained_vanilla_ohsumed_8_negs.pkl


In [13]:
for file in os.listdir("."):
    if ".pkl" in file and "cache" not in file and "vocab" not in file and "svm_coefficients_v1" not in file:
        
        f = open(file, "rb")
        class_vec = pickle.load(f)
        import random
        import time
        def zero_out(run):
            np.random.seed(int(time.time()))
            # run['query_vec'] = run['query_vec'].apply(lambda x: x * class_vec * 0)
            run["query_vec"] = [np.random.rand(30522) for _ in range(106)]
            return run

        p = lsr.query_encoder(sparse = False) >> pt.apply.generic(zero_out) >> index.quantized()
        df = pd.DataFrame({'qid': [str(i) for i in range(1, 107)]})

        # Generate random query vectors of length 30522 for each qid
        df['query_vec'] = [np.random.randint(1, 5001, size=30522) for _ in range(106)]
        fs = pt.Transformer.from_df(df)

        pipeline = fs >> index.quantized()
        print(pt.Experiment(
            [
                pt.io.read_results("./runs/splade_ohsumed_multiple_negative"),
                pt.io.read_results("./runs/splade_ohsumed_multiple_negative_svm_filter"),
                pt.io.read_results("./runs/splade_ohsumed_multiple_negative_t5_filter"),
                pipeline
            ],
            queries,
            qrels,
            eval_metrics=[nDCG@10, sens_docs@10],
            names = [
                "SPLADE",
                "SPLADE >> SVM Filter",
                "SPLADE >> T5 Filter",
                "Learned Vocab (Add)"
            ]
        ))

                   name   nDCG@10  sens_docs@10
0                SPLADE  0.437492      1.216981
1  SPLADE >> SVM Filter  0.360254      0.943396
2   SPLADE >> T5 Filter  0.428907      1.028302
3   Learned Vocab (Add)  0.188885      1.207547
                   name   nDCG@10  sens_docs@10
0                SPLADE  0.437492      1.216981
1  SPLADE >> SVM Filter  0.360254      0.943396
2   SPLADE >> T5 Filter  0.428907      1.028302
3   Learned Vocab (Add)  0.188885      1.207547
                   name   nDCG@10  sens_docs@10
0                SPLADE  0.437492      1.216981
1  SPLADE >> SVM Filter  0.360254      0.943396
2   SPLADE >> T5 Filter  0.428907      1.028302
3   Learned Vocab (Add)  0.188885      1.207547
                   name   nDCG@10  sens_docs@10
0                SPLADE  0.437492      1.216981
1  SPLADE >> SVM Filter  0.360254      0.943396
2   SPLADE >> T5 Filter  0.428907      1.028302
3   Learned Vocab (Add)  0.188885      1.207547
                   name   nDCG@10  sens_

KeyboardInterrupt: 

In [ ]:
retrieval_results = pt.Experiment(
    [
        pt.io.read_results("./runs/splade_ohsumed_multiple_negative"),
        pt.io.read_results("./runs/splade_ohsumed_multiple_negative_svm_filter"),
        pt.io.read_results("./runs/splade_ohsumed_multiple_negative_t5_filter"),
        pipeline
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = [
        "SPLADE",
        "SPLADE >> SVM Filter",
        "SPLADE >> T5 Filter",
        "Learned Vocab (Add)"
    ]
)

retrieval_results

In [ ]:
import sys
sys.path.append("./lsr_package")
from lsr.transformer import LSR
import pandas as pd
import torch

In [ ]:
model_folder = "/nfs/primary/SPLADE/splade_class/lsr_package/outputs/splade_ohsumed_multiple_negative/model/"
index_folder = "/nfs/primary/SPLADE/splade_class/indices/sens_reg_exp/splade_ohsumed_multiple_negative"
splade = LSR(model_folder).doc_encoder(sparse = False)

In [ ]:
training_docs = pd.read_pickle("/nfs/primary/sas_reranker/d2qmm_ohsumed_training_data.pkl")
texts = training_docs.text.tolist()
labels = training_docs.sensitivity.tolist()

In [ ]:
tensor = torch.stack([torch.tensor(arr) for arr in splade(training_docs).doc_vec.tolist()])
torch.save(tensor, "doc_vecs_ohsumed_8_negs.pt")